# codebook for testing seasonal variations code 

In [1]:
from utils_resampling import open_files

The resampled monthly mean for all four variables is now saved to the provided output path


/lustre/storeB/project/fou/hi/foccus/.venv/lib64/python3.11/site-packages/xarray/groupers.py:487: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  self.index_grouper = pd.Grouper(
/lustre/storeB/project/fou/hi/foccus/.venv/lib64/python3.11/site-packages/xarray/groupers.py:487: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  self.index_grouper = pd.Grouper(
/lustre/storeB/project/fou/hi/foccus/.venv/lib64/python3.11/site-packages/xarray/groupers.py:487: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  self.index_grouper = pd.Grouper(
/lustre/storeB/project/fou/hi/foccus/.venv/lib64/python3.11/site-packages/xarray/groupers.py:487: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  self.index_grouper = pd.Grouper(
/lustre/storeB/project/fou/hi/foccus/.venv/lib64/python3.11/site-package

The resampled yearly mean for all four variables is now saved to the provided output path
Please ensure that all seasons have a weight sum of 1.0 in the following statement! If not, refer to https://docs.xarray.dev/en/stable/examples/monthly-means.html.


UnboundLocalError: cannot access local variable 'weights' where it is not associated with a value

In [2]:
fl = '/lustre/storeB/project/fou/hi/foccus/malene/run-ocean-anemoi/ppi/external_checkpoint_inference/results/rollout_benchmark_lr3.5e6-rs3-i0-m3/step_51000_e14/resampled_means'
import xarray as xr
month = xr.open_dataset(f'{fl}/monthly_mean.nc') 
year = xr.open_dataset(f'{fl}/yearly_mean.nc')

In [4]:
def seasons(ds):
    #Then we make weights based on the number of days in the respective months
    month_length = ds.time.dt.days_in_month
    #Preview
    print(month_length)
    #Weights - tid per måned / summen av alle tider  
    weights = month_length.groupby('time.season') / month_length.groupby('time.season').sum(skipna = False)
    #testing to make sure the weights are correctly defined
    #weight_summed = np.testing.assert_allclose(weights.groupby('time.season').sum().values, np.ones(4)) 
    #if weight_summed != 1:
    #    raise ValueError(f'Weights are not properly defined - please ensure that the sum of weights for the seasons equal 1.0. Current sum is: {weight_summed}')
    #apply weights to dataset
    ds_weighted = (ds*weights).groupby('time.season').sum(dim = 'time', skipna = False)
    return ds_weighted 

In [5]:
ss = seasons(month)

<xarray.DataArray 'days_in_month' (time: 13)> Size: 104B
array([31, 29, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31, 31])
Coordinates:
  * time     (time) datetime64[ns] 104B 2024-01-01 2024-02-01 ... 2025-01-01


In [6]:
ss

<xarray.Dataset> Size: 505MB
Dimensions:        (season: 4, X: 1148, Y: 2747)
Coordinates:
  * X              (X) float64 9kB 0.0 800.0 1.6e+03 ... 9.168e+05 9.176e+05
  * Y              (Y) float64 22kB 0.0 800.0 1.6e+03 ... 2.196e+06 2.197e+06
  * season         (season) object 32B 'DJF' 'JJA' 'MAM' 'SON'
Data variables:
    salinity_0     (season, X, Y) float64 101MB nan nan nan ... 34.34 34.34
    temperature_0  (season, X, Y) float64 101MB nan nan nan ... 4.537 4.55 4.547
    u_eastward_0   (season, X, Y) float64 101MB 0.0001318 0.0002994 ... 0.02479
    v_northward_0  (season, X, Y) float64 101MB 1.847e-05 9.328e-05 ... 0.01471
    zeta           (season, X, Y) float64 101MB nan nan nan ... -0.2732 -0.2732

In [8]:
seasons = ss.to_netcdf(f'{fl}/seasonal_mean.nc')